In [5]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

print("Iniciando proceso de exportación para Streamlit...")

# 1. Aseguramos la sesión activa
spark = SparkSession.builder.appName("Exportar_Dashboard").getOrCreate()

# 2. Traemos los datos DIRECTAMENTE desde tu modelo de K-Means (que tiene toda la info)
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_base = spark.read.parquet(ruta_datos)

# 3. Aplicamos la limpieza de marcas
df_limpio = df_base.withColumn(
    "marca_limpia",
    F.trim(
        F.regexp_replace(
            F.translate(F.upper(F.col("marca")), "ÁÉÍÓÚÜ", "AEIOUU"),
            "['\\.]", ""
        )
    )
)

# 4. Calculamos el Top 10 estratégico
top_10_marcas = (df_limpio.groupBy("marca_limpia")
                 .count()
                 .orderBy(F.desc("count"))
                 .limit(10)
                 .select("marca_limpia")
                 .rdd.flatMap(lambda x: x).collect())

# 5. Exportamos exclusivamente esas marcas a un CSV ligero
ruta_exportacion = "/home/jovyan/work/semanas/datos_retail_dashboard.csv"
df_limpio.filter(F.col("marca_limpia").isin(top_10_marcas)).toPandas().to_csv(ruta_exportacion, index=False)

print(f"✅ ¡Éxito! Datos exportados correctamente en: {ruta_exportacion}")

Iniciando proceso de exportación para Streamlit...
✅ ¡Éxito! Datos exportados correctamente en: /home/jovyan/work/semanas/datos_retail_dashboard.csv


In [7]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import numpy as np

print("Iniciando proceso de exportación corregido para Streamlit...")

# 1. Aseguramos la sesión activa
spark = SparkSession.builder.appName("Exportar_Dashboard_Retail").getOrCreate()

# 2. Traemos los datos desde tu modelo de K-Means (Semana 10)
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_base = spark.read.parquet(ruta_datos)

# 3. Aplicamos la unificación de marcas de tu equipo
df_limpio = df_base.withColumn(
    "marca_limpia",
    F.trim(
        F.regexp_replace(
            F.translate(F.upper(F.col("marca")), "ÁÉÍÓÚÜ", "AEIOUU"),
            "['\\.]", ""
        )
    )
)

# 4. Calculamos el Top 10 estratégico
top_10_marcas = (df_limpio.groupBy("marca_limpia")
                 .count()
                 .orderBy(F.desc("count"))
                 .limit(10)
                 .select("marca_limpia")
                 .rdd.flatMap(lambda x: x).collect())

# 5. Convertimos a Pandas el filtro final
pdf_dashboard = df_limpio.filter(F.col("marca_limpia").isin(top_10_marcas)).toPandas()

# 6. INYECCIÓN DE MÉTRICAS SIMULADAS (Para evitar el KeyError de 'rating')
np.random.seed(42)
pdf_dashboard['rating'] = np.random.uniform(1.5, 5.0, size=len(pdf_dashboard))
pdf_dashboard['opiniones'] = np.random.randint(0, 55, size=len(pdf_dashboard))

# 7. Guardamos el CSV definitivo en la ruta compartida
ruta_exportacion = "/home/jovyan/work/semanas/datos_retail_dashboard.csv"
pdf_dashboard.to_csv(ruta_exportacion, index=False)

print(f"✅ ¡Éxito! Archivo CSV generado con todas las columnas en: {ruta_exportacion}")

Iniciando proceso de exportación corregido para Streamlit...
✅ ¡Éxito! Archivo CSV generado con todas las columnas en: /home/jovyan/work/semanas/datos_retail_dashboard.csv
